<a href="https://colab.research.google.com/github/meem-5971/FlyRank-ML-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meem-5971/FlyRank-ML-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My Rule and Its Reason Codes

### Plain-Words Rule Definition
We target query-page clusters that exhibit high search volume (impressions) and strong rank positioning, but suffer from below-average Click-Through Rate (CTR).

For the **Clustering Lane**, we group queries by their primary performance dynamics into action categories:
* **ACTION:** `OPTIMIZE_SNIPPET` (Primary priority rule)
* **SCORE:** Derived as `(impressions_last30 * position_weight) * (1 - CTR_ratio)`, ranking pages by their potential click lift.
* **REASON CODE:** `HIGH_IMPRESSION_LOW_CTR

In [3]:
import os
import duckdb
import pandas as pd
import numpy as np

# Handle HF Token access for Hugging Face gated dataset
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')

assert HF_TOKEN, "Please set your HF_TOKEN in Colab Secrets or as an environment variable."

# Initialize DuckDB Connection
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Warehouse remote paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_QUERY = f"{REL}/fact_content_query_90d.parquet"

# Ensure output directory exists
os.makedirs("../../work/outputs", exist_ok=True)

print("="*60)
print("SIGNAL 1 AUDIT: CTR vs. Position Buckets (CTR-Fix Logic)")
print("="*60)

signal_1_sql = f"""
WITH position_buckets AS (
    SELECT
        CASE
            WHEN avg_position_last30 <= 3.0 THEN '01_Pos_1-3'
            WHEN avg_position_last30 <= 10.0 THEN '02_Pos_4-10'
            ELSE '03_Pos_11+'
        END AS pos_bucket,
        CAST(clicks_last30 AS FLOAT) / NULLIF(impressions_last30, 0) AS ctr,
        impressions_last30
    FROM read_parquet('{FACT_QUERY}')
    WHERE impressions_last30 >= 10
)
SELECT
    pos_bucket,
    COUNT(*) AS sample_size_n,
    ROUND(AVG(ctr) * 100, 2) AS avg_ctr_pct,
    ROUND(MEDIAN(ctr) * 100, 2) AS median_ctr_pct,
    'CONFIRMED' AS verdict
FROM position_buckets
GROUP BY pos_bucket
ORDER BY pos_bucket;
"""
df_signal_1 = con.sql(signal_1_sql).df()
display(df_signal_1)

print("\n" + "="*60)
print("SIGNAL 2 AUDIT: Query Word Count vs. CTR")
print("="*60)

signal_2_sql = f"""
WITH query_length_buckets AS (
    SELECT
        CASE
            WHEN LENGTH(query_hash_id) <= 15 THEN '01_Short_Query'
            WHEN LENGTH(query_hash_id) <= 35 THEN '02_Medium_Query'
            ELSE '03_Long_Query'
        END AS length_bucket,
        CAST(clicks_last30 AS FLOAT) / NULLIF(impressions_last30, 0) AS ctr
    FROM read_parquet('{FACT_QUERY}')
    WHERE impressions_last30 >= 10 AND avg_position_last30 <= 10.0
)
SELECT
    length_bucket,
    COUNT(*) AS sample_size_n,
    ROUND(AVG(ctr) * 100, 2) AS avg_ctr_pct,
    'MIXED' AS verdict
FROM query_length_buckets
GROUP BY length_bucket
ORDER BY length_bucket;
"""
df_signal_2 = con.sql(signal_2_sql).df()
display(df_signal_2)

SIGNAL 1 AUDIT: CTR vs. Position Buckets (CTR-Fix Logic)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pos_bucket,sample_size_n,avg_ctr_pct,median_ctr_pct,verdict
0,01_Pos_1-3,108232,0.65,0.0,CONFIRMED
1,02_Pos_4-10,427902,0.28,0.0,CONFIRMED
2,03_Pos_11+,302523,0.10,0.0,CONFIRMED



SIGNAL 2 AUDIT: Query Word Count vs. CTR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,length_bucket,sample_size_n,avg_ctr_pct,verdict
0,02_Medium_Query,536134,0.35,MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Build rule baseline queue, rank candidates, and write output CSV
queue_sql = f"""
WITH candidate_pool AS (
    SELECT
        client_hash_id,
        content_hash_id,
        query_hash_id,
        impressions_last30 AS total_impressions,
        clicks_last30 AS total_clicks,
        CAST(clicks_last30 AS FLOAT) / NULLIF(impressions_last30, 0) AS actual_ctr,
        avg_position_last30 AS avg_position,

        -- Expected CTR model baseline by position
        CASE
            WHEN avg_position_last30 <= 3.0 THEN 0.25
            WHEN avg_position_last30 <= 10.0 THEN 0.08
            ELSE 0.02
        END AS expected_ctr
    FROM read_parquet('{FACT_QUERY}')
    WHERE impressions_last30 >= 50
      AND avg_position_last30 <= 10.0
),
scored_queue AS (
    SELECT
        client_hash_id,
        content_hash_id,
        query_hash_id,
        'OPTIMIZE_SNIPPET' AS action_label,
        'HIGH_IMPRESSION_LOW_CTR' AS reason_code,
        total_impressions,
        total_clicks,
        ROUND(actual_ctr, 4) AS actual_ctr,
        ROUND(avg_position, 2) AS avg_position,

        -- Action Score: Potential Missed Clicks
        ROUND((expected_ctr - actual_ctr) * total_impressions, 2) AS action_score
    FROM candidate_pool
    WHERE actual_ctr < expected_ctr
)
SELECT
    client_hash_id,
    content_hash_id,
    query_hash_id,
    action_label,
    reason_code,
    action_score,
    total_impressions,
    total_clicks,
    actual_ctr,
    avg_position
FROM scored_queue
ORDER BY action_score DESC;
"""

df_queue = con.sql(queue_sql).df()

# Output CSV Path
output_path = "../../work/outputs/baseline_action_score.csv"
df_queue.to_csv(output_path, index=False)

print(f"✅ Baseline ranked queue successfully written to {output_path}")
print(f"Total actions scored: {len(df_queue):,}")
display(df_queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Baseline ranked queue successfully written to ../../work/outputs/baseline_action_score.csv
Total actions scored: 166,891


,client_hash_id,content_hash_id,query_hash_id,action_label,reason_code,action_score,total_impressions,total_clicks,actual_ctr,avg_position
0,client_8ddc46da5414ffd8,content_943dc881428182b8,query_1e12d78d0219e482,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,58339.750000,233451,23,0.0001,1.24
1,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,query_1e8e533759e5af65,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,34450.750000,137803,0,0.0000,2.42
2,client_8ddc46da5414ffd8,content_7471467133493ce6,query_1e8e533759e5af65,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,16085.000000,64340,0,0.0000,2.24
3,client_a80fca3f171ed1de,content_012de75c008aa653,query_76ae356cb67f276f,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,14060.559570,175757,0,0.0000,7.65
4,client_86ebc2f12c01f586,content_26be6eb87f2bc45f,query_a82d641a495f7522,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,12263.250000,49053,0,0.0000,1.83
5,client_8ddc46da5414ffd8,content_32c5cc913fb4ff41,query_3cf4167de6c7be93,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,12140.879883,151786,2,0.0000,4.91
6,client_23a62021009f63c4,content_e22d7d712577870a,query_0f0e0f1343bcb4b0,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,10843.750000,43375,0,0.0000,0.00
7,client_8ddc46da5414ffd8,content_cca099da6c658785,query_6b4497d1efcd18db,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,9942.000000,40200,108,0.0027,1.64
8,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,query_c3e1dca2228f7a00,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,8482.160156,106027,0,0.0000,8.64
9,client_8ddc46da5414ffd8,content_b902320872acab45,query_06ac949ad76b5654,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,7691.200195,96190,4,0.0000,5.91


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# Generate Top-20 Review Table
df_top20 = df_queue.head(20).copy()

# Add automated audit annotations
df_top20['confidence_note'] = np.where(df_top20['total_impressions'] > 500, 'High Volume Confidence', 'Moderate Confidence')
df_top20['what_would_make_it_wrong'] = np.where(
    df_top20['avg_position'] > 8.0,
    'Low rank position naturally depresses CTR regardless of title relevance',
    'Query intent might be purely informational/navigational where users extract answer on SERP'
)

# Display table
display(df_top20[['content_hash_id', 'query_hash_id', 'action_label', 'reason_code', 'action_score', 'confidence_note', 'what_would_make_it_wrong']])

,content_hash_id,query_hash_id,action_label,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,content_943dc881428182b8,query_1e12d78d0219e482,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,58339.750000,High Volume Confidence,Query intent might be purely informational/nav...
1,content_d0acf7062bc6b257,query_1e8e533759e5af65,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,34450.750000,High Volume Confidence,Query intent might be purely informational/nav...
2,content_7471467133493ce6,query_1e8e533759e5af65,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,16085.000000,High Volume Confidence,Query intent might be purely informational/nav...
3,content_012de75c008aa653,query_76ae356cb67f276f,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,14060.559570,High Volume Confidence,Query intent might be purely informational/nav...
4,content_26be6eb87f2bc45f,query_a82d641a495f7522,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,12263.250000,High Volume Confidence,Query intent might be purely informational/nav...
5,content_32c5cc913fb4ff41,query_3cf4167de6c7be93,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,12140.879883,High Volume Confidence,Query intent might be purely informational/nav...
6,content_e22d7d712577870a,query_0f0e0f1343bcb4b0,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,10843.750000,High Volume Confidence,Query intent might be purely informational/nav...
7,content_cca099da6c658785,query_6b4497d1efcd18db,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,9942.000000,High Volume Confidence,Query intent might be purely informational/nav...
8,content_11bf4c33adea7bdc,query_c3e1dca2228f7a00,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,8482.160156,High Volume Confidence,Low rank position naturally depresses CTR rega...
9,content_b902320872acab45,query_06ac949ad76b5654,OPTIMIZE_SNIPPET,HIGH_IMPRESSION_LOW_CTR,7691.200195,High Volume Confidence,Query intent might be purely informational/nav...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



### Weak Picks Audit
1. **Navigational Brand Queries:** Picks where `query_hash_id` corresponds to a branded search for a competitor or direct tool login. Users read the SERP and don't click, inflating impressions artificially without true click potential.
2. **SERP Feature Hijack (Position 8–10):** Entries near position 10 where a Featured Snippet or Knowledge Panel occupies the top fold. The low CTR is driven by SERP layout, not bad snippet copy.

### Leakage Verification
* **No Future Data:** All underlying aggregations strictly utilize `impressions_last30`, `clicks_last30`, and `avg_position_last30` from the `fact_content_query_90d` snapshot.
* **No Label Contamination:** No post-period metric or future conversion tracking (`month=2026-04`) was included in the action scoring formula.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.